<a href="https://colab.research.google.com/github/mena-04/qml/blob/main/quantum_kernel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports

In [2]:

%pip install -q pennylane scikit-learn pandas matplotlib


In [3]:

import pennylane as qml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("PennyLane version:", qml.__version__)


PennyLane version: 0.45.1


Load classical dataset

In [4]:

from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()

zip_name = "qbreast_limited_data.zip"

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(".")

data_path = Path(
    "limited_data/data_n40_seed42.npz"
)

with np.load(data_path) as data:
    X_train = data["X_train"]
    X_test = data["X_test"]
    y_train = data["y_train"]
    y_test = data["y_test"]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)
print("Labels:", np.unique(y_train))

assert X_train.shape[1] == 4
assert set(np.unique(y_train)) == {0, 1}


Saving qbreast_limited_data.zip to qbreast_limited_data.zip
Training: (40, 4)
Testing: (114, 4)
Labels: [0 1]


Build the quantum feature map

In [5]:

N_QUBITS = 4

def feature_map(x):
    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)

    for i in range(N_QUBITS - 1):
        qml.CNOT(wires=[i, i + 1])

    for i in range(N_QUBITS):
        qml.RZ(x[i], wires=i)

dev = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

@qml.qnode(dev)
def quantum_state(x):
    feature_map(x)
    return qml.state()


Test the circuit

In [6]:

sample = X_train[0]

state = np.asarray(quantum_state(sample))

print("State shape:", state.shape)
print("State norm:", np.linalg.norm(state))

assert state.shape == (2 ** N_QUBITS,)
assert np.isclose(
    np.linalg.norm(state), 1.0,
    atol=1e-8
)

print(qml.draw(quantum_state)(sample))


State shape: (16,)
State norm: 1.0
0: ──RY(0.56)─╭●──RZ(0.56)─────────────────────┤  State
1: ──RY(1.51)─╰X─╭●─────────RZ(1.51)───────────┤  State
2: ──RY(0.71)────╰X────────╭●─────────RZ(0.71)─┤  State
3: ──RY(1.22)──────────────╰X─────────RZ(1.22)─┤  State


Compute the quantum kernel

In [7]:

def compute_states(X):
    return np.asarray([
        quantum_state(x) for x in X
    ], dtype=np.complex128)

def fidelity_kernel(states_a, states_b):
    overlaps = states_a @ states_b.conj().T
    return np.abs(overlaps) ** 2

start = time.perf_counter()

train_states = compute_states(X_train)
test_states = compute_states(X_test)

K_train = fidelity_kernel(
    train_states, train_states
)

K_test = fidelity_kernel(
    test_states, train_states
)

kernel_seconds = time.perf_counter() - start

print("Training kernel:", K_train.shape)
print("Testing kernel:", K_test.shape)
print("Kernel computation time:", kernel_seconds)


Training kernel: (40, 40)
Testing kernel: (114, 40)
Kernel computation time: 0.41412025199997515


Validate the kernel

In [8]:

print("Symmetric:",
      np.allclose(K_train, K_train.T))

print("Diagonal near 1:",
      np.allclose(np.diag(K_train), 1.0))

eigenvalues = np.linalg.eigvalsh(K_train)

print("Minimum eigenvalue:", eigenvalues.min())

assert np.allclose(
    K_train, K_train.T,
    atol=1e-8
)

assert np.allclose(
    np.diag(K_train), 1.0,
    atol=1e-8
)

assert eigenvalues.min() > -1e-8

print("Quantum kernel validation passed.")


Symmetric: True
Diagonal near 1: True
Minimum eigenvalue: 7.319982756733823e-05
Quantum kernel validation passed.


Train the Quantum Kernel SVM

In [9]:

quantum_svm = SVC(
    kernel="precomputed",
    C=1.0
)

start = time.perf_counter()

quantum_svm.fit(K_train, y_train)

svm_fit_seconds = time.perf_counter() - start

predictions = quantum_svm.predict(K_test)
scores = quantum_svm.decision_function(K_test)

print("SVM training time:", svm_fit_seconds)
print("Predictions:", np.unique(predictions, return_counts=True))


SVM training time: 0.0218916569999692
Predictions: (array([0, 1]), array([79, 35]))


Evaluate the quantum model

In [10]:

tn, fp, fn, tp = confusion_matrix(
    y_test, predictions,
    labels=[0, 1]
).ravel()

quantum_result = {
    "model": "Quantum Kernel SVM",
    "n_samples": 40,
    "seed": 42,
    "feature_map": "RY-CNOT-RZ",
    "C": 1.0,
    "accuracy": accuracy_score(
        y_test, predictions
    ),
    "precision": precision_score(
        y_test, predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test, predictions
    ),
    "f1": f1_score(
        y_test, predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test, scores
    ),
    "false_negatives": int(fn),
    "false_positives": int(fp),
    "kernel_seconds": kernel_seconds,
    "svm_fit_seconds": svm_fit_seconds
}

result_df = pd.DataFrame([quantum_result])

display(result_df.round(4))


,model,n_samples,seed,feature_map,C,accuracy,precision,recall,f1,roc_auc,false_negatives,false_positives,kernel_seconds,svm_fit_seconds
0,Quantum Kernel SVM,40,42,RY-CNOT-RZ,1.0,0.9211,0.9714,0.8095,0.8831,0.9921,8,1,0.4141,0.0219


Save the first quantum result

In [12]:

result_df.to_csv(
    "quantum_first_result.csv",
    index=False
)

print("Saved quantum_first_result.csv")


Saved quantum_first_result.csv


Tune Quantum SVM Regularization

In [13]:

from sklearn.model_selection import StratifiedKFold, GridSearchCV

cv_quantum = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

quantum_search = GridSearchCV(
    SVC(kernel="precomputed"),
    param_grid={"C": [0.1, 1, 10, 100]},
    scoring="roc_auc",
    cv=cv_quantum,
    refit=True,
    n_jobs=-1
)

quantum_search.fit(K_train, y_train)

print("Best quantum C:", quantum_search.best_params_)
print("CV ROC-AUC:", quantum_search.best_score_)

tuned_quantum = quantum_search.best_estimator_

tuned_scores = tuned_quantum.decision_function(K_test)
tuned_pred = tuned_quantum.predict(K_test)

print("Test recall:",
      recall_score(y_test, tuned_pred))
print("Test F1:",
      f1_score(y_test, tuned_pred))
print("Test ROC-AUC:",
      roc_auc_score(y_test, tuned_scores))


Best quantum C: {'C': 10}
CV ROC-AUC: 1.0
Test recall: 0.8333333333333334
Test F1: 0.8860759493670886
Test ROC-AUC: 0.9837962962962963


Compare the tuned and untuned quantum models

In [14]:

comparison_rows = []

for name, model in [
    ("Quantum SVM C=1", quantum_svm),
    ("Quantum SVM Tuned C", tuned_quantum)
]:
    pred = model.predict(K_test)
    scores = model.decision_function(K_test)

    tn, fp, fn, tp = confusion_matrix(
        y_test, pred
    ).ravel()

    comparison_rows.append({
        "model": name,
        "recall": recall_score(y_test, pred),
        "precision": precision_score(
            y_test, pred, zero_division=0
        ),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, scores),
        "false_negatives": fn,
        "false_positives": fp
    })

quantum_comparison = pd.DataFrame(comparison_rows)

display(quantum_comparison.round(4))


,model,recall,precision,f1,roc_auc,false_negatives,false_positives
0,Quantum SVM C=1,0.8095,0.9714,0.8831,0.9921,8,1
1,Quantum SVM Tuned C,0.8333,0.9459,0.8861,0.9838,7,2


Training-Only Threshold Calibration for Quantum SVM

In [15]:

from sklearn.model_selection import cross_val_predict
from sklearn.base import clone

# Use the tuned quantum model configuration
quantum_for_cv = SVC(
    kernel="precomputed",
    C=quantum_search.best_params_["C"]
)

# Generate out-of-fold scores manually because
# precomputed kernels need fold-specific slicing.
oof_scores = np.zeros(len(y_train))

for train_fold, val_fold in cv_quantum.split(
    K_train, y_train
):
    K_fold_train = K_train[
        np.ix_(train_fold, train_fold)
    ]

    K_fold_val = K_train[
        np.ix_(val_fold, train_fold)
    ]

    fold_model = clone(quantum_for_cv)
    fold_model.fit(
        K_fold_train,
        y_train[train_fold]
    )

    oof_scores[val_fold] = (
        fold_model.decision_function(K_fold_val)
    )

# Find threshold that achieves >= 95% OOF recall
threshold_candidates = []

for threshold in np.unique(oof_scores):

    pred = (oof_scores >= threshold).astype(int)

    rec = recall_score(y_train, pred)

    tn, fp, fn, tp = confusion_matrix(
        y_train, pred
    ).ravel()

    specificity = tn / (tn + fp)

    if rec >= 0.95:
        threshold_candidates.append(
            (specificity, threshold)
        )

if threshold_candidates:
    _, quantum_threshold = max(
        threshold_candidates,
        key=lambda x: (x[0], x[1])
    )
else:
    quantum_threshold = 0.0

print("Selected quantum threshold:", quantum_threshold)
print(
    "OOF recall:",
    recall_score(
        y_train,
        (oof_scores >= quantum_threshold).astype(int)
    )
)


Selected quantum threshold: -0.6913910025039102
OOF recall: 1.0


Evaluate the calibrated quantum SVM

In [16]:

# Test scores from the already tuned quantum model
test_scores = tuned_quantum.decision_function(K_test)

calibrated_predictions = (
    test_scores >= quantum_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test, calibrated_predictions
).ravel()

calibrated_quantum_result = {
    "model": "Quantum SVM (Threshold Adjusted)",
    "n_samples": 40,
    "seed": 42,
    "C": quantum_search.best_params_["C"],
    "threshold": quantum_threshold,
    "accuracy": accuracy_score(
        y_test, calibrated_predictions
    ),
    "precision": precision_score(
        y_test, calibrated_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test, calibrated_predictions
    ),
    "f1": f1_score(
        y_test, calibrated_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test, test_scores
    ),
    "false_negatives": int(fn),
    "false_positives": int(fp)
}

calibrated_quantum_df = pd.DataFrame(
    [calibrated_quantum_result]
)

display(calibrated_quantum_df.round(4))


,model,n_samples,seed,C,threshold,accuracy,precision,recall,f1,roc_auc,false_negatives,false_positives
0,Quantum SVM (Threshold Adjusted),40,42,10,-0.6914,0.9474,0.9091,0.9524,0.9302,0.9838,2,4


Multi-Seed Quantum Kernel Evaluation

In [17]:

from pathlib import Path
import time

TRAIN_SIZES = [40, 80, 160]
SEEDS = [42, 123, 2026]

quantum_records = []

for n_samples in TRAIN_SIZES:
    for seed in SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as d:
            Xtr = d["X_train"].copy()
            Xte = d["X_test"].copy()
            ytr = d["y_train"].copy()
            yte = d["y_test"].copy()

        start = time.perf_counter()

        states_tr = compute_states(Xtr)
        states_te = compute_states(Xte)

        Ktr = fidelity_kernel(
            states_tr, states_tr
        )
        Kte = fidelity_kernel(
            states_te, states_tr
        )

        kernel_time = time.perf_counter() - start

        # Fixed C for the first controlled experiment
        model = SVC(
            kernel="precomputed",
            C=1.0
        )

        model.fit(Ktr, ytr)

        pred = model.predict(Kte)
        scores = model.decision_function(Kte)

        tn, fp, fn, tp = confusion_matrix(
            yte, pred,
            labels=[0, 1]
        ).ravel()

        quantum_records.append({
            "model": "Quantum Kernel SVM",
            "n_samples": n_samples,
            "seed": seed,
            "feature_map": "RY-CNOT-RZ",
            "C": 1.0,
            "recall": recall_score(yte, pred),
            "precision": precision_score(
                yte, pred, zero_division=0
            ),
            "f1": f1_score(
                yte, pred, zero_division=0
            ),
            "roc_auc": roc_auc_score(yte, scores),
            "false_negatives": int(fn),
            "false_positives": int(fp),
            "kernel_seconds": kernel_time
        })

        print(
            f"n={n_samples}, seed={seed}: "
            f"recall={recall_score(yte, pred):.4f}, "
            f"AUC={roc_auc_score(yte, scores):.4f}"
        )

quantum_all = pd.DataFrame(quantum_records)

quantum_all.to_csv(
    "quantum_all_results.csv",
    index=False
)

display(quantum_all.round(4))


n=40, seed=42: recall=0.8095, AUC=0.9921
n=40, seed=123: recall=0.6905, AUC=0.9778
n=40, seed=2026: recall=0.6667, AUC=0.9934
n=80, seed=42: recall=0.7619, AUC=0.9944
n=80, seed=123: recall=0.6905, AUC=0.9911
n=80, seed=2026: recall=0.8095, AUC=0.9884
n=160, seed=42: recall=0.8333, AUC=0.9940
n=160, seed=123: recall=0.7857, AUC=0.9957
n=160, seed=2026: recall=0.8333, AUC=0.9983


,model,n_samples,seed,feature_map,C,recall,precision,f1,roc_auc,false_negatives,false_positives,kernel_seconds
0,Quantum Kernel SVM,40,42,RY-CNOT-RZ,1.0,0.8095,0.9714,0.8831,0.9921,8,1,0.6615
1,Quantum Kernel SVM,40,123,RY-CNOT-RZ,1.0,0.6905,0.9667,0.8056,0.9778,13,1,0.8445
2,Quantum Kernel SVM,40,2026,RY-CNOT-RZ,1.0,0.6667,1.0000,0.8000,0.9934,14,0,0.7722
3,Quantum Kernel SVM,80,42,RY-CNOT-RZ,1.0,0.7619,0.9697,0.8533,0.9944,10,1,0.8537
4,Quantum Kernel SVM,80,123,RY-CNOT-RZ,1.0,0.6905,0.9667,0.8056,0.9911,13,1,0.6648
5,Quantum Kernel SVM,80,2026,RY-CNOT-RZ,1.0,0.8095,1.0000,0.8947,0.9884,8,0,0.9116
6,Quantum Kernel SVM,160,42,RY-CNOT-RZ,1.0,0.8333,0.9722,0.8974,0.9940,7,1,1.1999
7,Quantum Kernel SVM,160,123,RY-CNOT-RZ,1.0,0.7857,0.9706,0.8684,0.9957,9,1,1.1662
8,Quantum Kernel SVM,160,2026,RY-CNOT-RZ,1.0,0.8333,1.0000,0.9091,0.9983,7,0,1.1560


Quantum Performance Summary

In [18]:

quantum_summary = (
    quantum_all
    .groupby("n_samples")
    .agg(
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        auc_mean=("roc_auc", "mean"),
        auc_std=("roc_auc", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        kernel_time_mean=("kernel_seconds", "mean")
    )
    .reset_index()
)

display(quantum_summary.round(4))

quantum_summary.to_csv(
    "quantum_summary.csv",
    index=False
)


,n_samples,recall_mean,recall_std,auc_mean,auc_std,f1_mean,f1_std,kernel_time_mean
0,40,0.7222,0.0765,0.9878,0.0086,0.8296,0.0465,0.7594
1,80,0.7540,0.0599,0.9913,0.0030,0.8512,0.0446,0.8101
2,160,0.8175,0.0275,0.9960,0.0022,0.8916,0.0209,1.1740


Training-Only Quantum Threshold Calibration

In [19]:

from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from pathlib import Path
import numpy as np
import pandas as pd
import time

calibrated_quantum_records = []

for n_samples in [40, 80, 160]:
    for seed in [42, 123, 2026]:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as d:
            Xtr = d["X_train"].copy()
            Xte = d["X_test"].copy()
            ytr = d["y_train"].copy()
            yte = d["y_test"].copy()

        start = time.perf_counter()

        # Compute quantum kernel
        states_tr = compute_states(Xtr)
        states_te = compute_states(Xte)

        Ktr = fidelity_kernel(states_tr, states_tr)
        Kte = fidelity_kernel(states_te, states_tr)

        # Out-of-fold predictions
        cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        oof_scores = np.zeros(len(ytr))

        for train_fold, val_fold in cv.split(Ktr, ytr):

            K_fold_train = Ktr[
                np.ix_(train_fold, train_fold)
            ]

            K_fold_val = Ktr[
                np.ix_(val_fold, train_fold)
            ]

            fold_model = SVC(
                kernel="precomputed",
                C=1.0
            )

            fold_model.fit(
                K_fold_train,
                ytr[train_fold]
            )

            oof_scores[val_fold] = (
                fold_model.decision_function(K_fold_val)
            )

        # Select threshold targeting >=95% OOF recall
        candidates = []

        for threshold in np.unique(oof_scores):

            pred = (oof_scores >= threshold).astype(int)

            recall = recall_score(ytr, pred)

            tn, fp, fn, tp = confusion_matrix(
                ytr, pred, labels=[0, 1]
            ).ravel()

            specificity = tn / (tn + fp)

            if recall >= 0.95:
                candidates.append(
                    (specificity, threshold)
                )

        if candidates:
            _, selected_threshold = max(
                candidates,
                key=lambda x: (x[0], x[1])
            )
        else:
            selected_threshold = 0.0

        # Train final model on entire training subset
        model = SVC(
            kernel="precomputed",
            C=1.0
        )

        model.fit(Ktr, ytr)

        test_scores = model.decision_function(Kte)

        predictions = (
            test_scores >= selected_threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            yte, predictions, labels=[0, 1]
        ).ravel()

        calibrated_quantum_records.append({
            "model": "Quantum SVM (Adjusted)",
            "n_samples": n_samples,
            "seed": seed,
            "C": 1.0,
            "threshold": selected_threshold,
            "accuracy": accuracy_score(yte, predictions),
            "precision": precision_score(
                yte, predictions, zero_division=0
            ),
            "recall": recall_score(yte, predictions),
            "f1": f1_score(
                yte, predictions, zero_division=0
            ),
            "roc_auc": roc_auc_score(yte, test_scores),
            "false_negatives": int(fn),
            "false_positives": int(fp),
            "runtime_seconds": time.perf_counter() - start
        })

        print(
            f"n={n_samples}, seed={seed}: "
            f"recall={recall_score(yte, predictions):.4f}, "
            f"FP={fp}, FN={fn}"
        )

calibrated_quantum_df = pd.DataFrame(
    calibrated_quantum_records
)

display(calibrated_quantum_df.round(4))

calibrated_quantum_df.to_csv(
    "quantum_threshold_adjusted.csv",
    index=False
)


n=40, seed=42: recall=0.9762, FP=4, FN=1
n=40, seed=123: recall=1.0000, FP=18, FN=0
n=40, seed=2026: recall=1.0000, FP=5, FN=0
n=80, seed=42: recall=1.0000, FP=2, FN=0
n=80, seed=123: recall=0.9524, FP=1, FN=2
n=80, seed=2026: recall=0.9286, FP=3, FN=3
n=160, seed=42: recall=1.0000, FP=3, FN=0
n=160, seed=123: recall=0.9286, FP=1, FN=3
n=160, seed=2026: recall=0.9762, FP=1, FN=1


,model,n_samples,seed,C,threshold,accuracy,precision,recall,f1,roc_auc,false_negatives,false_positives,runtime_seconds
0,Quantum SVM (Adjusted),40,42,1.0,-0.5850,0.9561,0.9111,0.9762,0.9425,0.9921,1,4,0.8951
1,Quantum SVM (Adjusted),40,123,1.0,-0.7856,0.8421,0.7000,1.0000,0.8235,0.9778,0,18,0.9346
2,Quantum SVM (Adjusted),40,2026,1.0,-0.5956,0.9561,0.8936,1.0000,0.9438,0.9934,0,5,0.8791
3,Quantum SVM (Adjusted),80,42,1.0,-0.5660,0.9825,0.9545,1.0000,0.9767,0.9944,0,2,1.4540
4,Quantum SVM (Adjusted),80,123,1.0,-0.5842,0.9737,0.9756,0.9524,0.9639,0.9911,2,1,1.2632
5,Quantum SVM (Adjusted),80,2026,1.0,-0.3866,0.9474,0.9286,0.9286,0.9286,0.9884,3,3,1.2510
6,Quantum SVM (Adjusted),160,42,1.0,-0.4466,0.9737,0.9333,1.0000,0.9655,0.9940,0,3,3.5442
7,Quantum SVM (Adjusted),160,123,1.0,-0.4323,0.9649,0.9750,0.9286,0.9512,0.9957,3,1,2.5783
8,Quantum SVM (Adjusted),160,2026,1.0,-0.3574,0.9825,0.9762,0.9762,0.9762,0.9983,1,1,2.1567


Aggregate Quantum Threshold Results

In [20]:

quantum_adjusted_summary = (
    calibrated_quantum_df
    .groupby("n_samples")
    .agg(
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        auc_mean=("roc_auc", "mean"),
        fn_mean=("false_negatives", "mean"),
        fp_mean=("false_positives", "mean")
    )
    .reset_index()
)

display(quantum_adjusted_summary.round(4))

quantum_adjusted_summary.to_csv(
    "quantum_adjusted_summary.csv",
    index=False
)


,n_samples,recall_mean,recall_std,precision_mean,f1_mean,auc_mean,fn_mean,fp_mean
0,40,0.9921,0.0137,0.8349,0.9033,0.9878,0.3333,9.0000
1,80,0.9603,0.0364,0.9529,0.9564,0.9913,1.6667,2.0000
2,160,0.9683,0.0364,0.9615,0.9643,0.9960,1.3333,1.6667


Classical vs Quantum Threshold Comparison

In [24]:

from google.colab import files
from pathlib import Path

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded:
    print(filename)

print("CSV files available:")
for path in Path(".").rglob("*.csv"):
    print(path)

import pandas as pd

classical_adjusted = pd.read_csv(
    "rbf_threshold_adjusted.csv"
)

quantum_adjusted = pd.read_csv(
    "quantum_threshold_adjusted.csv"
)

comparison = classical_adjusted.merge(
    quantum_adjusted,
    on=["n_samples", "seed"],
    suffixes=("_classical", "_quantum"),
    validate="one_to_one"
)

assert len(comparison) == 9

comparison["recall_difference"] = (
    comparison["recall_quantum"] -
    comparison["recall_classical"]
)

comparison["f1_difference"] = (
    comparison["f1_quantum"] -
    comparison["f1_classical"]
)

comparison["fp_difference"] = (
    comparison["false_positives_quantum"] -
    comparison["false_positives_classical"]
)

display(comparison[[
    "n_samples",
    "seed",
    "recall_classical",
    "recall_quantum",
    "recall_difference",
    "f1_difference",
    "fp_difference"
]].round(4))


Saving rbf_threshold_adjusted.csv to rbf_threshold_adjusted.csv
Uploaded files:
rbf_threshold_adjusted.csv
CSV files available:
quantum_summary.csv
rbf_threshold_adjusted.csv
quantum_first_result.csv
quantum_adjusted_summary.csv
quantum_all_results.csv
quantum_threshold_adjusted.csv
limited_data/classical_summary.csv
limited_data/classical_limited_results.csv
sample_data/mnist_train_small.csv
sample_data/mnist_test.csv
sample_data/california_housing_train.csv
sample_data/california_housing_test.csv


,n_samples,seed,recall_classical,recall_quantum,recall_difference,f1_difference,fp_difference
0,40,42,1.0000,0.9762,-0.0238,0.0092,-2
1,40,123,0.8810,1.0000,0.1190,-0.0789,15
2,40,2026,0.9524,1.0000,0.0476,0.0243,0
3,80,42,1.0000,1.0000,0.0000,0.0434,-4
4,80,123,0.8810,0.9524,0.0714,0.0389,0
5,80,2026,0.8810,0.9286,0.0476,0.0036,2
6,160,42,0.9762,1.0000,0.0238,0.0008,1
7,160,123,0.9762,0.9286,-0.0476,-0.0250,0
8,160,2026,0.9524,0.9762,0.0238,0.0123,0


Entangling vs Non-Entangling Quantum Feature Maps

In [25]:

import pennylane as qml
import numpy as np

N_QUBITS = 4

dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev)
def state_no_entanglement(x):
    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)
    return qml.state()

@qml.qnode(dev)
def state_entangling(x):
    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)

    for i in range(N_QUBITS - 1):
        qml.CNOT(wires=[i, i + 1])

    return qml.state()

def kernel_from_states(Xa, Xb, state_fn):
    states_a = np.asarray(
        [state_fn(x) for x in Xa],
        dtype=np.complex128
    )
    states_b = np.asarray(
        [state_fn(x) for x in Xb],
        dtype=np.complex128
    )
    return np.abs(states_a @ states_b.conj().T) ** 2


Check whether the kernels differ

In [26]:

from pathlib import Path

with np.load(
    "limited_data/data_n40_seed42.npz"
) as d:
    X_demo = d["X_train"][:10].copy()

K_simple = kernel_from_states(
    X_demo, X_demo, state_no_entanglement
)

K_ent = kernel_from_states(
    X_demo, X_demo, state_entangling
)

print("Maximum kernel difference:",
      np.max(np.abs(K_simple - K_ent)))

print("Kernels identical:",
      np.allclose(K_simple, K_ent))


Maximum kernel difference: 4.440892098500626e-16
Kernels identical: True


Build a meaningful entangling feature map

In [27]:

@qml.qnode(dev)
def state_data_reuploading(x):
    # First encoding layer
    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)

    # Entangling layer
    for i in range(N_QUBITS - 1):
        qml.CNOT(wires=[i, i + 1])

    # Second data-dependent encoding layer
    for i in range(N_QUBITS):
        qml.RZ(x[i], wires=i)
        qml.RY(x[i], wires=i)

    return qml.state()

K_reupload = kernel_from_states(
    X_demo, X_demo, state_data_reuploading
)

print("Difference from simple kernel:",
      np.max(np.abs(K_simple - K_reupload)))

print("Difference from old entangling kernel:",
      np.max(np.abs(K_ent - K_reupload)))


Difference from simple kernel: 0.3301774027506809
Difference from old entangling kernel: 0.330177402750681


Leakage-safe preprocessing for cross-validation

In [28]:

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

data = load_breast_cancer()
X_raw = data.data
y_all = (data.target == 0).astype(int)

def make_fold_preprocessor():
    return Pipeline([
        ("standardize", StandardScaler()),
        ("pca", PCA(n_components=4)),
        ("angles", MinMaxScaler(
            feature_range=(0, np.pi),
            clip=True
        ))
    ])


Compare the two feature maps

In [29]:

from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

with np.load(
    "limited_data/data_n40_seed42.npz"
) as d:
    subset_idx = d["train_idx"].copy()

X_subset_raw = X_raw[subset_idx]
y_subset = y_all[subset_idx]

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

feature_maps = {
    "Simple RY": state_no_entanglement,
    "Data Reuploading": state_data_reuploading
}

records = []

for name, state_fn in feature_maps.items():

    fold_aucs = []

    for train_fold, val_fold in cv.split(
        X_subset_raw, y_subset
    ):
        prep = make_fold_preprocessor()

        X_fold_train = prep.fit_transform(
            X_subset_raw[train_fold]
        )

        X_fold_val = prep.transform(
            X_subset_raw[val_fold]
        )

        K_train_fold = kernel_from_states(
            X_fold_train,
            X_fold_train,
            state_fn
        )

        K_val_fold = kernel_from_states(
            X_fold_val,
            X_fold_train,
            state_fn
        )

        model = SVC(
            kernel="precomputed",
            C=1.0
        )

        model.fit(
            K_train_fold,
            y_subset[train_fold]
        )

        scores = model.decision_function(
            K_val_fold
        )

        auc = roc_auc_score(
            y_subset[val_fold],
            scores
        )

        fold_aucs.append(auc)

    records.append({
        "feature_map": name,
        "cv_auc_mean": np.mean(fold_aucs),
        "cv_auc_std": np.std(fold_aucs, ddof=1)
    })

feature_map_results = pd.DataFrame(records)

display(feature_map_results.round(4))


,feature_map,cv_auc_mean,cv_auc_std
0,Simple RY,0.9750,0.0433
1,Data Reuploading,0.9528,0.0210


Verify Quantum Kernel Against Classical Closed Form

In [30]:

def classical_ry_kernel(Xa, Xb):
    differences = (
        Xa[:, None, :] - Xb[None, :, :]
    )

    return np.prod(
        np.cos(differences / 2) ** 2,
        axis=2
    )

K_quantum = kernel_from_states(
    X_demo,
    X_demo,
    state_no_entanglement
)

K_classical = classical_ry_kernel(
    X_demo,
    X_demo
)

print(
    "Maximum difference:",
    np.max(np.abs(K_quantum - K_classical))
)

assert np.allclose(
    K_quantum,
    K_classical,
    atol=1e-10
)

print("Classical and quantum kernels match.")


Maximum difference: 1.3322676295501878e-15
Classical and quantum kernels match.


Leakage-Safe Quantum Feature-Map Benchmark

In [31]:

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC
from pathlib import Path
import numpy as np
import pandas as pd
import time

FEATURE_MAPS = {
    "Simple RY": state_no_entanglement,
    "Data Reuploading": state_data_reuploading
}

C_VALUES = [0.1, 1, 10]
TRAIN_SIZES = [40, 80, 160]
SEEDS = [42, 123, 2026]

selection_records = []
final_records = []


Compare feature maps using training-only CV

In [32]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

for n_samples in TRAIN_SIZES:
    for seed in SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as d:
            subset_idx = d["train_idx"].copy()
            test_idx_local = d["test_idx"].copy()

        X_subset_raw = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        X_test_raw_local = X_raw[test_idx_local]
        y_test_local = y_all[test_idx_local]

        cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        best_auc = -np.inf
        best_config = None

        for map_name, state_fn in FEATURE_MAPS.items():

            for C in C_VALUES:

                fold_aucs = []

                for train_fold, val_fold in cv.split(
                    X_subset_raw, y_subset
                ):
                    prep = make_fold_preprocessor()

                    X_fold_train = prep.fit_transform(
                        X_subset_raw[train_fold]
                    )

                    X_fold_val = prep.transform(
                        X_subset_raw[val_fold]
                    )

                    K_train_fold = kernel_from_states(
                        X_fold_train,
                        X_fold_train,
                        state_fn
                    )

                    K_val_fold = kernel_from_states(
                        X_fold_val,
                        X_fold_train,
                        state_fn
                    )

                    model = SVC(
                        kernel="precomputed",
                        C=C
                    )

                    model.fit(
                        K_train_fold,
                        y_subset[train_fold]
                    )

                    scores = model.decision_function(
                        K_val_fold
                    )

                    fold_aucs.append(
                        roc_auc_score(
                            y_subset[val_fold],
                            scores
                        )
                    )

                mean_auc = np.mean(fold_aucs)

                selection_records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "feature_map": map_name,
                    "C": C,
                    "cv_auc": mean_auc
                })

                if mean_auc > best_auc:
                    best_auc = mean_auc
                    best_config = (
                        map_name, C, state_fn
                    )

        # Fit selected configuration on full subset
        map_name, best_C, state_fn = best_config

        final_prep = make_fold_preprocessor()

        Xtr = final_prep.fit_transform(
            X_subset_raw
        )
        Xte = final_prep.transform(
            X_test_raw_local
        )

        start = time.perf_counter()

        Ktr = kernel_from_states(
            Xtr, Xtr, state_fn
        )
        Kte = kernel_from_states(
            Xte, Xtr, state_fn
        )

        kernel_seconds = time.perf_counter() - start

        model = SVC(
            kernel="precomputed",
            C=best_C
        )

        model.fit(Ktr, y_subset)

        pred = model.predict(Kte)
        scores = model.decision_function(Kte)

        tn, fp, fn, tp = confusion_matrix(
            y_test_local, pred,
            labels=[0, 1]
        ).ravel()

        final_records.append({
            "n_samples": n_samples,
            "seed": seed,
            "selected_map": map_name,
            "selected_C": best_C,
            "cv_auc": best_auc,
            "recall": recall_score(
                y_test_local, pred
            ),
            "precision": precision_score(
                y_test_local, pred,
                zero_division=0
            ),
            "f1": f1_score(
                y_test_local, pred,
                zero_division=0
            ),
            "roc_auc": roc_auc_score(
                y_test_local, scores
            ),
            "false_negatives": int(fn),
            "false_positives": int(fp),
            "kernel_seconds": kernel_seconds
        })

        print(
            f"n={n_samples}, seed={seed}: "
            f"selected={map_name}, C={best_C}, "
            f"AUC={roc_auc_score(y_test_local, scores):.4f}"
        )

selection_df = pd.DataFrame(selection_records)
final_feature_map_df = pd.DataFrame(final_records)

display(final_feature_map_df.round(4))


n=40, seed=42: selected=Simple RY, C=10, AUC=0.9914
n=40, seed=123: selected=Data Reuploading, C=10, AUC=0.9590
n=40, seed=2026: selected=Simple RY, C=1, AUC=0.9947
n=80, seed=42: selected=Simple RY, C=0.1, AUC=0.9940
n=80, seed=123: selected=Simple RY, C=1, AUC=0.9917
n=80, seed=2026: selected=Simple RY, C=1, AUC=0.9934
n=160, seed=42: selected=Simple RY, C=1, AUC=0.9967
n=160, seed=123: selected=Simple RY, C=10, AUC=0.9970
n=160, seed=2026: selected=Simple RY, C=0.1, AUC=0.9977


,n_samples,seed,selected_map,selected_C,cv_auc,recall,precision,f1,roc_auc,false_negatives,false_positives,kernel_seconds
0,40,42,Simple RY,10.0,0.9917,0.8571,0.9730,0.9114,0.9914,6,1,0.4105
1,40,123,Data Reuploading,10.0,0.9833,0.7381,0.8857,0.8052,0.9590,11,4,1.2216
2,40,2026,Simple RY,1.0,0.9556,0.7143,1.0000,0.8333,0.9947,12,0,0.4687
3,80,42,Simple RY,0.1,0.9804,0.0952,1.0000,0.1739,0.9940,38,0,0.5194
4,80,123,Simple RY,1.0,0.9940,0.7143,0.9677,0.8219,0.9917,12,1,0.7245
5,80,2026,Simple RY,1.0,0.9917,0.8095,1.0000,0.8947,0.9934,8,0,0.5154
6,160,42,Simple RY,1.0,0.9864,0.9048,0.9744,0.9383,0.9967,4,1,1.1984
7,160,123,Simple RY,10.0,0.9819,0.9048,0.9744,0.9383,0.9970,4,1,0.7828
8,160,2026,Simple RY,0.1,0.9890,0.5952,1.0000,0.7463,0.9977,17,0,0.9039


Analyze Selected Quantum Architectures

In [33]:

print("Selected feature maps:")
print(final_feature_map_df["selected_map"].value_counts())

summary = (
    final_feature_map_df
    .groupby("n_samples")
    .agg(
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        auc_mean=("roc_auc", "mean"),
        auc_std=("roc_auc", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std")
    )
    .reset_index()
)

display(summary.round(4))

selection_df.to_csv(
    "quantum_feature_map_cv.csv",
    index=False
)

final_feature_map_df.to_csv(
    "quantum_feature_map_final.csv",
    index=False
)

summary.to_csv(
    "quantum_feature_map_summary.csv",
    index=False
)


Selected feature maps:
selected_map
Simple RY           8
Data Reuploading    1
Name: count, dtype: int64


,n_samples,recall_mean,recall_std,auc_mean,auc_std,f1_mean,f1_std
0,40,0.7698,0.0765,0.9817,0.0197,0.8500,0.0550
1,80,0.5397,0.3878,0.9931,0.0012,0.6302,0.3968
2,160,0.8016,0.1787,0.9971,0.0005,0.8743,0.1109


Final Benchmark Configuration

In [34]:

import numpy as np
import pandas as pd
import time
import json

from pathlib import Path
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, accuracy_score, confusion_matrix
)

# Load raw data
dataset = load_breast_cancer()
X_raw = dataset.data
y_all = (dataset.target == 0).astype(int)

TRAIN_SIZES = [40, 80, 160]
SEEDS = [42, 123, 2026]
C_VALUES = [0.1, 1, 10]

FEATURE_MAPS = {
    "Simple RY": state_no_entanglement,
    "Data Reuploading": state_data_reuploading
}

def make_preprocessor():
    return Pipeline([
        ("standardize", StandardScaler()),
        ("pca", PCA(n_components=4)),
        ("angles", MinMaxScaler(
            feature_range=(0, np.pi),
            clip=True
        ))
    ])

print("Final benchmark configuration loaded.")


Final benchmark configuration loaded.


Classical and Quantum Kernel Computation

In [35]:

from sklearn.metrics.pairwise import rbf_kernel

def compute_kernel(Xa, Xb, model_type, config):

    if model_type == "classical":
        return rbf_kernel(
            Xa, Xb,
            gamma=config["gamma"]
        )

    if model_type == "quantum":
        state_fn = FEATURE_MAPS[
            config["feature_map"]
        ]

        return kernel_from_states(
            Xa, Xb, state_fn
        )

    raise ValueError("Unknown model type")


Training-Only Model Selection

In [36]:

from itertools import product

def get_configs(model_type):

    if model_type == "classical":
        return [
            {"C": C, "gamma": gamma}
            for C, gamma in product(
                C_VALUES,
                [0.1, 1, "scale"]
            )
        ]

    return [
        {"C": C, "feature_map": name}
        for C, name in product(
            C_VALUES,
            FEATURE_MAPS.keys()
        )
    ]


def select_config(X, y, model_type, seed):

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=seed
    )

    best_auc = -np.inf
    best_config = None

    for config in get_configs(model_type):

        fold_aucs = []

        for tr, val in cv.split(X, y):

            prep = make_preprocessor()

            Xtr = prep.fit_transform(X[tr])
            Xval = prep.transform(X[val])

            # Resolve gamma='scale' on this fold
            cfg = config.copy()

            if (
                model_type == "classical"
                and cfg["gamma"] == "scale"
            ):
                cfg["gamma"] = (
                    1.0 /
                    (Xtr.shape[1] * Xtr.var())
                )

            Ktr = compute_kernel(
                Xtr, Xtr, model_type, cfg
            )

            Kval = compute_kernel(
                Xval, Xtr, model_type, cfg
            )

            model = SVC(
                kernel="precomputed",
                C=config["C"]
            )

            model.fit(Ktr, y[tr])

            scores = model.decision_function(Kval)

            fold_aucs.append(
                roc_auc_score(y[val], scores)
            )

        mean_auc = np.mean(fold_aucs)

        if mean_auc > best_auc:
            best_auc = mean_auc
            best_config = config.copy()

    return best_config, best_auc


Unified Model Training

In [37]:

def fit_and_score(
    X_train_raw,
    y_train,
    X_eval_raw,
    model_type,
    config
):
    prep = make_preprocessor()

    Xtr = prep.fit_transform(X_train_raw)
    Xeval = prep.transform(X_eval_raw)

    cfg = config.copy()

    if (
        model_type == "classical"
        and cfg["gamma"] == "scale"
    ):
        cfg["gamma"] = (
            1.0 /
            (Xtr.shape[1] * Xtr.var())
        )

    Ktr = compute_kernel(
        Xtr, Xtr, model_type, cfg
    )

    Keval = compute_kernel(
        Xeval, Xtr, model_type, cfg
    )

    model = SVC(
        kernel="precomputed",
        C=config["C"]
    )

    model.fit(Ktr, y_train)

    return model.decision_function(Keval)


Leakage-Safe Threshold Selection

In [38]:

def select_threshold(X, y, model_type, seed):

    outer_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=seed
    )

    oof_scores = np.zeros(len(y))

    for fold_num, (tr, val) in enumerate(
        outer_cv.split(X, y)
    ):

        # Nested model selection
        config, _ = select_config(
            X[tr],
            y[tr],
            model_type,
            seed + fold_num
        )

        # Fit only on outer training fold
        oof_scores[val] = fit_and_score(
            X[tr],
            y[tr],
            X[val],
            model_type,
            config
        )

    # Choose threshold targeting >=95% recall
    candidates = []

    for threshold in np.unique(oof_scores):

        pred = (
            oof_scores >= threshold
        ).astype(int)

        rec = recall_score(y, pred)

        tn, fp, fn, tp = confusion_matrix(
            y, pred, labels=[0, 1]
        ).ravel()

        specificity = tn / (tn + fp)

        if rec >= 0.95:
            candidates.append(
                (specificity, threshold)
            )

    if candidates:
        _, threshold = max(
            candidates,
            key=lambda x: (x[0], x[1])
        )
    else:
        threshold = 0.0

    return threshold


Final Classical vs Quantum Evaluation

In [39]:

final_records = []

for n_samples in TRAIN_SIZES:
    for seed in SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as d:
            subset_idx = d["train_idx"].copy()
            test_idx = d["test_idx"].copy()

        Xtr_raw = X_raw[subset_idx]
        ytr = y_all[subset_idx]

        Xte_raw = X_raw[test_idx]
        yte = y_all[test_idx]

        for model_type in ["classical", "quantum"]:

            print(
                f"Running {model_type}: "
                f"n={n_samples}, seed={seed}"
            )

            start = time.perf_counter()

            # Nested training-only threshold calibration
            threshold = select_threshold(
                Xtr_raw,
                ytr,
                model_type,
                seed
            )

            # Select final model using full training subset
            config, cv_auc = select_config(
                Xtr_raw,
                ytr,
                model_type,
                seed
            )

            # Fit final model and evaluate on test set
            scores = fit_and_score(
                Xtr_raw,
                ytr,
                Xte_raw,
                model_type,
                config
            )

            pred = (
                scores >= threshold
            ).astype(int)

            tn, fp, fn, tp = confusion_matrix(
                yte, pred, labels=[0, 1]
            ).ravel()

            final_records.append({
                "model": model_type,
                "n_samples": n_samples,
                "seed": seed,
                "config": json.dumps(config),
                "cv_auc": cv_auc,
                "threshold": threshold,
                "accuracy": accuracy_score(yte, pred),
                "precision": precision_score(
                    yte, pred, zero_division=0
                ),
                "recall": recall_score(yte, pred),
                "f1": f1_score(
                    yte, pred, zero_division=0
                ),
                "roc_auc": roc_auc_score(yte, scores),
                "false_negatives": int(fn),
                "false_positives": int(fp),
                "runtime_seconds": (
                    time.perf_counter() - start
                )
            })

            # Save progress after each model
            pd.DataFrame(final_records).to_csv(
                "final_fair_comparison.csv",
                index=False
            )

final_df = pd.DataFrame(final_records)

display(final_df.round(4))


Running classical: n=40, seed=42
Running quantum: n=40, seed=42
Running classical: n=40, seed=123
Running quantum: n=40, seed=123
Running classical: n=40, seed=2026
Running quantum: n=40, seed=2026
Running classical: n=80, seed=42
Running quantum: n=80, seed=42
Running classical: n=80, seed=123
Running quantum: n=80, seed=123
Running classical: n=80, seed=2026
Running quantum: n=80, seed=2026
Running classical: n=160, seed=42
Running quantum: n=160, seed=42
Running classical: n=160, seed=123
Running quantum: n=160, seed=123
Running classical: n=160, seed=2026
Running quantum: n=160, seed=2026


,model,n_samples,seed,config,cv_auc,threshold,accuracy,precision,recall,f1,roc_auc,false_negatives,false_positives,runtime_seconds
0,classical,40,42,"{""C"": 1, ""gamma"": 0.1}",0.9917,-0.9303,0.8596,0.7241,1.0000,0.8400,0.9934,0,16,1.6845
1,quantum,40,42,"{""C"": 10, ""feature_map"": ""Simple RY""}",0.9917,-0.8832,0.9211,0.8235,1.0000,0.9032,0.9914,0,9,27.9769
2,classical,40,123,"{""C"": 10, ""gamma"": 0.1}",0.9833,-0.9598,0.9211,0.8235,1.0000,0.9032,0.9861,0,9,1.1174
3,quantum,40,123,"{""C"": 10, ""feature_map"": ""Data Reuploading""}",0.9833,-0.9323,0.8421,0.7069,0.9762,0.8200,0.9590,1,17,14.8875
4,classical,40,2026,"{""C"": 0.1, ""gamma"": 0.1}",0.9704,-0.9259,0.9386,0.8889,0.9524,0.9195,0.9947,2,5,1.1542
5,quantum,40,2026,"{""C"": 1, ""feature_map"": ""Simple RY""}",0.9556,-0.8657,0.7982,0.6462,1.0000,0.7850,0.9947,0,23,15.6047
6,classical,80,42,"{""C"": 0.1, ""gamma"": 0.1}",0.9882,-0.8991,0.9474,0.8750,1.0000,0.9333,0.9957,0,6,1.7160
7,quantum,80,42,"{""C"": 0.1, ""feature_map"": ""Simple RY""}",0.9804,-0.7761,0.9474,0.8913,0.9762,0.9318,0.9940,1,5,29.0735
8,classical,80,123,"{""C"": 0.1, ""gamma"": 0.1}",0.9898,-0.8099,0.9386,0.9730,0.8571,0.9114,0.9947,6,1,1.2498
9,quantum,80,123,"{""C"": 1, ""feature_map"": ""Simple RY""}",0.9940,-0.4319,0.9649,0.9524,0.9524,0.9524,0.9917,2,2,25.8209


Aggregate Classical vs Quantum Results

In [40]:

final_summary = (
    final_df
    .groupby(["model", "n_samples"])
    .agg(
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        auc_mean=("roc_auc", "mean"),
        auc_std=("roc_auc", "std"),
        fn_mean=("false_negatives", "mean"),
        fp_mean=("false_positives", "mean"),
        runtime_mean=("runtime_seconds", "mean")
    )
    .reset_index()
)

display(final_summary.round(4))

final_summary.to_csv(
    "final_fair_summary.csv",
    index=False
)


,model,n_samples,recall_mean,recall_std,precision_mean,f1_mean,auc_mean,auc_std,fn_mean,fp_mean,runtime_mean
0,classical,40,0.9841,0.0275,0.8122,0.8876,0.9914,0.0046,0.6667,10.0000,1.3187
1,classical,80,0.9444,0.0765,0.8516,0.8882,0.9906,0.0079,2.3333,8.0000,1.4362
2,classical,160,0.9683,0.0364,0.9421,0.9536,0.9974,0.0003,1.3333,2.6667,1.7394
3,quantum,40,0.9921,0.0137,0.7255,0.8361,0.9817,0.0197,0.3333,16.3333,19.4897
4,quantum,80,0.9683,0.0137,0.8774,0.9188,0.9931,0.0012,1.3333,6.0000,26.9332
5,quantum,160,0.9286,0.0630,0.9764,0.9505,0.9971,0.0005,3.0000,1.0000,61.6833
